In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch        
import torchvision
import random

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [7]:
df = pd.read_csv("seattle-weather.csv")
df.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


In [13]:
df["date"] = pd.to_datetime(df["date"])
df

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain
...,...,...,...,...,...,...
1456,2015-12-27,8.6,4.4,1.7,2.9,rain
1457,2015-12-28,1.5,5.0,1.7,1.3,rain
1458,2015-12-29,0.0,7.2,0.6,2.6,fog
1459,2015-12-30,0.0,5.6,-1.0,3.4,sun


In [3]:
protected = ["date", "weather"]
feature_cols = [c for c in df.columns if c not in protected]

In [8]:
station_sensors = {
    "A": ["temp_min", "temp_max"],
    "B": ["precipitation"],
    "C": ["wind"]
}

In [9]:
stations = {}

for name, cols in station_sensors.items():
    stations[name] = df[protected + cols].copy()

# Baseline Centralized

In [27]:
X_train = train_df[["precipitation","temp_max","temp_min","wind"]]
X_test = test_df[["precipitation","temp_max","temp_min","wind"]]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

central = LogisticRegression(max_iter=500)
central.fit(X_train, y_train)

print("Centralized Accuracy:",
      accuracy_score(y_test, central.predict(X_test)))

Centralized Accuracy: 0.7815699658703071


# Method 1: Model Fusion (Stacking)

In [14]:
le = LabelEncoder()
df["weather_enc"] = le.fit_transform(df["weather"])

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["weather_enc"])

In [17]:
def get_station_data(data, cols):
    X = data[cols].values
    y = data["weather_enc"].values
    return X, y

In [18]:
local_models = {}
train_probs = []
test_probs = []

for name, cols in station_sensors.items():
    X_train, y_train = get_station_data(train_df, cols)
    X_test, y_test = get_station_data(test_df, cols)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = LogisticRegression(max_iter=500)
    model.fit(X_train, y_train)

    local_models[name] = (model, scaler)

    train_probs.append(model.predict_proba(X_train))
    test_probs.append(model.predict_proba(X_test))

In [19]:
meta_X_train = np.hstack(train_probs)
meta_X_test = np.hstack(test_probs)

In [20]:
meta_model = LogisticRegression(max_iter=500)
meta_model.fit(meta_X_train, y_train)

meta_preds = meta_model.predict(meta_X_test)

In [22]:
print("Strategy 1 Accuracy:",
      accuracy_score(y_test, meta_preds))

Strategy 1 Accuracy: 0.8122866894197952


# Method 2: Embedding Share

In [24]:
embedding_models = {}
train_embeddings = []
test_embeddings = []

for name, cols in station_sensors.items():
    X_train, y_train = get_station_data(train_df, cols)
    X_test, y_test = get_station_data(test_df, cols)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = MLPClassifier(hidden_layer_sizes=(16,), max_iter=500)
    model.fit(X_train, y_train)

    embedding_models[name] = (model, scaler)

    # Extract hidden layer output
    train_embed = model._predict(X_train)
    test_embed = model._predict(X_test)

    train_embeddings.append(train_embed)
    test_embeddings.append(test_embed)

/Users/vivekrai/opt/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [25]:
fusion_X_train = np.hstack(train_embeddings)
fusion_X_test = np.hstack(test_embeddings)

In [26]:
fusion_model = LogisticRegression(max_iter=500)
fusion_model.fit(fusion_X_train, y_train)

fusion_preds = fusion_model.predict(fusion_X_test)

print("Strategy 3 Accuracy:",
      accuracy_score(y_test, fusion_preds))

ValueError: Expected 2D array, got 1D array instead:
array=[4. 4. 2. ... 4. 4. 4.].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.